In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate hf_transfer

In [ ]:
# hf_transfer: tải model đa luồng, nhanh hơn hẳn client mặc định.
# Set biến môi trường "HF_TOKEN" khi tạo pod (RunPod: mục "Environment Variables" lúc deploy) —
# KHÔNG hardcode token vào file này, vì file nằm trong git repo.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
else:
    print("[WARN] Chưa set biến môi trường 'HF_TOKEN' → dùng unauthenticated request tới HF Hub (có thể bị rate limit).")


In [ ]:
import json
from pathlib import Path

import torch
from peft import PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

## Helper functions — đọc/ghi dữ liệu

In [ ]:
def _get_records(jsonl_path: Path) -> list[dict]:
    """
    - Summary: Đọc toàn bộ record hợp lệ từ file JSONL.
    - Args:
        - jsonl_path: Đường dẫn file JSONL.
    - Output:
        - list[dict]: Danh sách record hợp lệ.
    """
    records: list[dict] = []
    with jsonl_path.open(encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except Exception:
                    pass
    return records


def _get_existing_ids(out_path: Path) -> set[str]:
    """
    - Summary: Đọc file output, trả về tập id đã xử lý.
    - Args:
        - out_path: Đường dẫn file output JSONL.
    - Output:
        - set[str]: Tập hợp id đã tồn tại (để resume, bỏ qua khi infer lại).
    """
    if not out_path.exists():
        return set()
    return {record.get("id") for record in _get_records(out_path)}


def _append_jsonl(record: dict, out_path: Path):
    """
    - Summary: Append 1 record ra file JSONL, tạo file nếu chưa có.
    - Args:
        - record: Record cần ghi ({id, predict}).
        - out_path: Đường dẫn file output JSONL.
    - Output:
        - None. Ghi thêm 1 dòng vào out_path.
    """
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')


def _get_prompt_prefix(prompt_path: Path | None) -> str:
    """
    - Summary: Đọc nội dung prompt cần chèn trước content.
    - Args:
        - prompt_path: Đường dẫn file prompt, None nếu không dùng.
    - Output:
        - str: Nội dung prompt, "" nếu prompt_path là None.
    """
    if prompt_path is None:
        return ""
    return prompt_path.read_text(encoding='utf-8')

## Helper functions — prompt & sinh prediction

In [ ]:
def _build_messages(content: str, prompt_prefix: str) -> list[dict]:
    """
    - Summary: Build list message theo chat template Qwen.
    - Args:
        - content: Nội dung văn bản cần trích xuất sự kiện.
        - prompt_prefix: Prompt hệ thống chèn trước content, "" nếu không có.
    - Output:
        - list[dict]: List message [system?, user].
    """
    messages: list[dict] = []
    if prompt_prefix:
        messages.append({"role": "system", "content": prompt_prefix})
    messages.append({"role": "user", "content": content})
    return messages


def _get_model_prediction(model, tokenizer, content: str, prompt_prefix: str, max_new_tokens: int) -> str:
    """
    - Summary: Sinh prediction cho 1 sample bằng model đã gắn adapter.
    - Args:
        - model: Model gốc đã gắn adapter LoRA.
        - tokenizer: Tokenizer tương ứng.
        - content: Nội dung văn bản cần trích xuất sự kiện.
        - prompt_prefix: Prompt hệ thống chèn trước content.
        - max_new_tokens: Số token tối đa cần sinh.
    - Output:
        - str: Chuỗi raw model sinh ra (chưa parse JSON), để tự tính độ đo sau.
    """
    messages    = _build_messages(content, prompt_prefix)
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs      = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            pad_token_id   = tokenizer.pad_token_id,
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

## Model + adapter (load để infer)

In [ ]:
def _get_base_model_name(adapter_dir: Path) -> str:
    """
    - Summary: Đọc tên model gốc từ adapter_config.json.
    - Args:
        - adapter_dir: Thư mục chứa adapter weight đã train.
    - Output:
        - str: Tên model gốc trên HF Hub.
    """
    adapter_config = json.loads((adapter_dir / "adapter_config.json").read_text(encoding="utf-8"))
    return adapter_config["base_model_name_or_path"]


def _build_model_and_tokenizer_for_infer(adapter_dir: Path):
    """
    - Summary:
        1. Đọc tên model gốc từ adapter_config.json (_get_base_model_name()).
        2. Load tokenizer + model gốc ở dạng 4-bit.
        3. Gắn adapter LoRA đã train từ adapter_dir.
    - Args:
        - adapter_dir: Thư mục chứa adapter weight đã train.
    - Output:
        - tuple: (model đã gắn adapter, tokenizer).
    """
    model_name = _get_base_model_name(adapter_dir)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit              = True,
        bnb_4bit_quant_type       = "nf4",
        bnb_4bit_compute_dtype    = torch.bfloat16,
        bnb_4bit_use_double_quant = True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token  # Qwen không có pad_token riêng

    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config = bnb_config,
        device_map          = "auto",
    )
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()
    model.config.use_cache = True
    return model, tokenizer

## Orchestrator: infer từng file, nhiều file

In [ ]:
def _process_file(
    model,
    tokenizer,
    in_path:        Path,
    out_path:       Path,
    prompt_prefix:  str,
    max_new_tokens: int,
):
    """
    - Summary:
        1. Tải id đã xử lý, để resume (_get_existing_ids()).
        2. Đọc record từ input, bỏ id đã xử lý (_get_records()).
        3. Sinh prediction từng record (_get_model_prediction()).
        4. Append từng prediction ra output (_append_jsonl()).
    - Args:
        - model: Model đã gắn adapter, dùng để infer.
        - tokenizer: Tokenizer tương ứng.
        - in_path: Đường dẫn file JSONL input.
        - out_path: Đường dẫn file JSONL output.
        - prompt_prefix: Prompt hệ thống chèn trước content.
        - max_new_tokens: Số token tối đa cần sinh.
    - Output:
        - None. Ghi predict liên tục ra out_path.
    """
    existing_ids = _get_existing_ids(out_path)
    records      = [r for r in _get_records(in_path) if r.get("id") not in existing_ids]

    print(f"[INFER] {in_path.name}: {len(records)} sample cần infer (đã có {len(existing_ids)})")
    for i, record in enumerate(records, start=1):
        predict = _get_model_prediction(model, tokenizer, record["content"], prompt_prefix, max_new_tokens)
        _append_jsonl({"id": record.get("id"), "predict": predict}, out_path)
        if i % 20 == 0:
            print(f"[INFER] {in_path.name}: {i}/{len(records)}")


def run_inference(
    dir:            Path | str,
    in_out:         list[tuple[Path | str, Path | str]],
    prompt_path:    Path | None = None,
    max_new_tokens: int = 512,
):
    """
    - Summary:
        1. Build model + tokenizer gắn adapter (_build_model_and_tokenizer_for_infer()).
        2. Đọc prompt prefix (_get_prompt_prefix()).
        3. Infer và ghi từng cặp input/output (_process_file()).
    - Args:
        - dir: Thư mục chứa adapter weight đã train.
        - in_out: List cặp (đường dẫn input, đường dẫn output).
        - prompt_path: Đường dẫn file prompt chèn trước content, None nếu không dùng.
        - max_new_tokens: Số token tối đa cần sinh khi infer.
    - Output:
        - None. Predict được ghi liên tục vào từng file output tương ứng.
    """
    adapter_dir      = Path(dir)
    prompt_prefix    = _get_prompt_prefix(prompt_path)
    model, tokenizer = _build_model_and_tokenizer_for_infer(adapter_dir)

    for in_path, out_path in in_out:
        _process_file(
            model          = model,
            tokenizer      = tokenizer,
            in_path        = Path(in_path),
            out_path       = Path(out_path),
            prompt_prefix  = prompt_prefix,
            max_new_tokens = max_new_tokens,
        )

## Config & chạy

Sửa `dir` (thư mục weight) và `in_out` (cặp input/output) theo môi trường đang chạy.

In [ ]:
config = {
    "dir": "/workspace/output/train_v1/Qwen_Qwen2.5-3B-Instruct_train_v1_16",
    "in_out": [
        ["/workspace/val_v1.jsonl",  "/workspace/output/predict/val_v1_predict.jsonl"],
        ["/workspace/test_v1.jsonl", "/workspace/output/predict/test_v1_predict.jsonl"],
    ],
    "prompt_path": "/workspace/study_prompt.v2.txt",  # None nếu không chèn prompt
    "max_new_tokens": 512,
}

adapter_dir    = config["dir"]
in_out_paths   = config["in_out"]
prompt_path    = Path(config["prompt_path"]) if config["prompt_path"] else None
max_new_tokens = config["max_new_tokens"]

In [ ]:
run_inference(
    dir            = adapter_dir,
    in_out         = in_out_paths,
    prompt_path    = prompt_path,
    max_new_tokens = max_new_tokens,
)